# Rate model outputs — SonarQube + MLScent

Rates the Code4ML answers of every evaluated run with the same SonarQube + MLScent calls used for
`code4ml_annotated.csv` (`experiment/rating/pipeline/rate_annotated.py`, imported from there), then uploads the ratings next to the run on the Hub.

    HF dimsmauls/code4ml-eval-outputs/<run>/code4ml_outputs.jsonl
      -> CSV (id, code_block, ...)
      -> SonarQube (:8000 / :8001) + MLScent (:8101 / :8102), two halves in parallel
      -> <run>/code4ml_ratings.csv on the Hub

- Only rows with `ast_valid = True` are rated (AST validity gate); the other rows stay in the CSV with empty ratings.
- No MLScent notebook context: model answers do not come from a Kaggle notebook, so `context_imports` is empty and
  `code_blocks.csv` is never read.
- Resumable: a run already on the Hub is skipped; an interrupted run resumes from its `part_1.csv` / `part_2.csv`.

**Before running:** start both SonarQube API containers (`sonar_api1`, `sonar_api2`) and both MLScent containers
(`mlscent-1`, `mlscent-2`). The kernel needs `httpx` and `huggingface_hub`, and a Hugging Face login with write access.

## 1. Configuration

In [2]:
import csv
import json
import sys
import threading
from collections import Counter
from pathlib import Path

import httpx
from huggingface_hub import HfApi, hf_hub_download

HERE = Path.cwd().resolve()                                   # experiment/experiment/rating (this notebook)
RATING_PIPELINE = HERE.parents[1] / "rating" / "pipeline"     # experiment/rating/pipeline: rate_annotated.py
assert (RATING_PIPELINE / "rate_annotated.py").exists(), f"rate_annotated.py not found in {RATING_PIPELINE}"
sys.path.insert(0, str(RATING_PIPELINE))
from rate_annotated import merge, mlscent_sender, rate_half, sonar_sender, split_halves  # noqa: E402

OUTPUT_REPO = "dimsmauls/code4ml-eval-outputs"
RUNS = ["vanilla", "few-shot", "qwen-14b"] + [f"{m}/seed-{s}" for m in ("lora", "qlora") for s in (42, 43, 44)]
SONAR = ["http://localhost:8000", "http://localhost:8001"]
MLSCENT = ["http://localhost:8101", "http://localhost:8102"]
BATCH_SIZE = 50
WORK_DIR = HERE / "results"
RATINGS_FILE = "code4ml_ratings.csv"

## 2. Check the four services

In [3]:
for url in SONAR + MLSCENT:
    httpx.get(f"{url}/health", timeout=10).raise_for_status()
print("SonarQube x2 and MLScent x2 are up")

SonarQube x2 and MLScent x2 are up


## 3. Steps

In [4]:
api = HfApi()


def to_csv(run):
    """Download <run>/code4ml_outputs.jsonl and write it as the CSV rate_annotated expects (id, code_block)."""
    path = hf_hub_download(OUTPUT_REPO, f"{run}/code4ml_outputs.jsonl", repo_type="dataset")
    with open(path) as handle:
        rows = [json.loads(line) for line in handle]
    out = WORK_DIR / run.replace("/", "_") / RATINGS_FILE
    out.parent.mkdir(parents=True, exist_ok=True)
    fields = ["id", "graph_vertex_class", "graph_vertex_subclass", "ast_valid", "code_block"]
    with out.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({"id": row["id"], "graph_vertex_class": row["graph_vertex_class"],
                             "graph_vertex_subclass": row["graph_vertex_subclass"],
                             "ast_valid": row["ast_valid"], "code_block": row["code"]})
    return out


def rate(csv_path):
    """Rate the AST-valid rows on both instance pairs in parallel, then merge the ratings into csv_path."""
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if row["ast_valid"] == "True"]
    parts = [csv_path.parent / "part_1.csv", csv_path.parent / "part_2.csv"]
    errors = []

    def work(half, sonar_url, mlscent_url, out):
        try:  # no kernel_of / imports: MLScent gets no notebook context
            rate_half(half, {}, out, sonar_sender(sonar_url), mlscent_sender(mlscent_url), BATCH_SIZE)
        except BaseException as exc:  # noqa: BLE001 - reported after both threads stop
            errors.append(exc)

    threads = [threading.Thread(target=work, args=args)
               for args in zip(split_halves(rows), SONAR, MLSCENT, parts)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise RuntimeError(f"rating failed (rerun to resume): {errors}")
    merge(csv_path, parts, require_all=False)  # AST-invalid rows keep empty ratings

## 4. Run

In [5]:
on_hub = set(api.list_repo_files(OUTPUT_REPO, repo_type="dataset"))
for run in RUNS:
    if f"{run}/{RATINGS_FILE}" in on_hub:
        print(f"{run}: already on the Hub, skipping")
        continue
    csv_path = to_csv(run)
    rate(csv_path)
    api.upload_file(path_or_fileobj=str(csv_path), path_in_repo=f"{run}/{RATINGS_FILE}", repo_id=OUTPUT_REPO,
                    repo_type="dataset", commit_message=f"rating {run}")
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rated = [row for row in csv.DictReader(handle) if row["ast_valid"] == "True"]
    print(f"{run}: {len(rated)} rated | combined_rating {dict(sorted(Counter(r['combined_rating'] for r in rated).items()))}")
print(f"done: https://huggingface.co/datasets/{OUTPUT_REPO}")

part_2.csv: 50/149
part_1.csv: 50/149
part_1.csv: 100/149
part_2.csv: 100/149
part_2.csv: 149/149
part_1.csv: 149/149
vanilla: 298 rated | combined_rating {'A': 157, 'B': 70, 'C': 14, 'D': 34, 'N/A': 23}
part_1.csv: 50/152
part_2.csv: 50/152
part_2.csv: 100/152
part_1.csv: 100/152
part_2.csv: 150/152
part_1.csv: 150/152
part_2.csv: 152/152
part_1.csv: 152/152
few-shot: 304 rated | combined_rating {'A': 11, 'B': 29, 'C': 2, 'D': 14, 'N/A': 248}
part_1.csv: 50/152
part_2.csv: 50/152
part_2.csv: 100/152
part_1.csv: 100/152
part_2.csv: 150/152
part_1.csv: 150/152
part_2.csv: 152/152
part_1.csv: 152/152
qwen-14b: 304 rated | combined_rating {'A': 163, 'B': 42, 'C': 47, 'D': 23, 'N/A': 29}
part_1.csv: 50/151
part_2.csv: 50/150
part_1.csv: 100/151
part_2.csv: 100/150
part_1.csv: 150/151
part_2.csv: 150/150
part_1.csv: 151/151
lora/seed-42: 301 rated | combined_rating {'A': 19, 'B': 56, 'N/A': 226}
part_2.csv: 50/141
part_1.csv: 50/142
part_2.csv: 100/141
part_1.csv: 100/142
part_2.csv: 141/14